Main credit:  https://github.com/christianversloot/machine-learning-articles/blob/main/how-to-build-a-resnet-from-scratch-with-tensorflow-2-and-keras.md

*Thank you to the author Christian Versloot (link above) for sharing such incredible in-depth knowledge for free!*

*This note is for me personally to study ResNet, summarize and add details. For full study, use the link above.*


ResNet is commonly used as a backbone in current neural network models.

Issues prior to ResNet:
- In theory, the deeper the network, the better as it should be able to learn all the feature representations.
- However even with activation functions such as nonlinear ReLU and Batch Normalization, <u>issues with vanishing and exploding gradients </u>  persisted.
- Shattering gradients, where neural network gradients resemble white noise during optimization may be the issue.

ResNet novelty in:
- Residual learning: Skip connection
 - Skip connection such that deeper layers have direct connection to upper layers, effectively reducing vanishing and exploding gradient problems;
 - allow gradients (rate of change of loss function with respect to model's weights and biases) to flow directly through the network during backpropagation.
 - Instead of learning full mapping `H(x)`, ResNet layers learn residual functions `F(x)=H(x)-x`, allowing it to be easier in optimization.
 - By focus on learning residuals than direct mapping, ResNet avoids redundant feature learning (improve parameter efficiency, leading to better generalization and performance).
- Identity/Projection mapping
 - Each residual block learns the difference between its input and output. (Ensures they have the same size)
 - With residual learning, identity mappings is more easily learnt, <u>avoiding degradation problem</u> where deeper networks perform worse than shallow.

Implementation of the skip connection can be simply added to the output of the regular block, however this may produce <u>issues related to dimensionality and feature map (width and height of mask) size </u>.

This is resolved by using:
- <b>Identity mapping</b>, maps the input to output, adding padding or reducing feature map size where necessary.
- <b>Projection mapping</b>, uses convolution to generate an output that <u>has the same size</u> with the next residual building block.


References:
Original paper: https://openaccess.thecvf.com/content_cvpr_2016/papers/He_Deep_Residual_Learning_CVPR_2016_paper.pdf


More info: https://machinecurve.com/index.php/2022/01/13/resnet-a-simple-introduction#introducing-residual-networks-resnets

------------------------------
Changed to fix error or to notebook compatible:
1. model_configuration - `49 checkpoint - filepath`
2. preprocessed_dataset - `21 train_generator` - `import tensorflow to tf` somehow fixes `AttributeError: module 'keras.api.applications.resnet50' has no attribute 'preprocessing_input'` 🤷

Note: Is due to outdated implementation. Using `import tensorflow to tf` prevents any conflict.

Note: `import tensorflow as tf`, `from tensorflow.keras import Model`, you can see the conflict happening if you do not change the prefix as industrial standard.

3. Preprocess data - `tf.math.floor` to use `math.floor` to fix  
`AttributeError: 'numpy.float32' object has no attribute 'index'`

###Import Libraries

In [ ]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.layers import Add, GlobalAveragePooling2D,\
	Dense, Flatten, Conv2D, Lambda,	Input, BatchNormalization, Activation
from tensorflow.keras.optimizers import schedules, SGD
from tensorflow.keras.callbacks import TensorBoard, ModelCheckpoint

- `tensorflow.keras import Model` - Using Model class initiator from keras.
- `tensorflow.keras.datasets` - Using keras' cifar10 dataset
- `tensorflow.keras.layers` - Import other important layers such as Conv2D,  BatchNormalization, Activation, etc.
- `tensorflow.keras.optimizer` - Using keras' implementation of standard optimizer.
- `tensorflow.keras.callbacks` - To visualize and save the model checkpoints.

###Model configuration

In [ ]:
import math
def model_configuration():
  # Load dataset for computing dataset size
  (input_train, _), (_,_) = load_dataset()

  # Generic configuration
  width, height, channels = 32, 32, 3
  batch_size = 128
  num_classes = 10
  validation_split = 0.1 # 45/5 per He et al. paper
  verbose = 1
  n = 9
  init_fm_dim = 16
  shortcut_type = "identity" # or: projection

  # Dataset size
  train_size = (1-validation_split) * len(input_train)
  val_size = (validation_split) * len(input_train)

  # Number of steps per epoch is dependent on batch size
  maximum_number_iterations = 64000 # per He et al. paper
  steps_per_epoch = math.floor(train_size / batch_size)
  val_steps_per_epoch = math.floor(val_size / batch_size)
  epochs = tf.cast(math.floor(maximum_number_iterations / steps_per_epoch), dtype=tf.int64)

  # Define loss function
  loss = tf.keras.losses.CategoricalCrossentropy(from_logits=True)

  # Learning rate configuraiton per He et al. paper
  boundaries = [32000, 48000]
  values = [0.1, 0.01, 0.001]
  lr_schedule = schedules.PiecewiseConstantDecay(boundaries, values)

  # Set layer init
  initializer = tf.keras.initializers.HeNormal()

  # Define optimizer
  optimizer_momentum = 0.9
  optimizer_additional_metrics = ["accuracy"]
  optimizer = SGD(learning_rate=lr_schedule, momentum=optimizer_momentum)

  # Load Tensorboard callback
  tensorboard = TensorBoard(
	  log_dir=os.path.join(os.getcwd(), "logs"),
	  histogram_freq=1,
	  write_images=True
	)

	# Save a model checkpoint after every epoch
  checkpoint = ModelCheckpoint(
		filepath='/content/model_checkpoint.h5',
    save_best_only=True,
		save_freq="epoch"
	)

	# Add callbacks to list
  callbacks = [
	  tensorboard,
	  checkpoint
	]

	# Create config dictionary
  config = {
		"width": width,
		"height": height,
		"dim": channels,
		"batch_size": batch_size,
		"num_classes": num_classes,
		"validation_split": validation_split,
		"verbose": verbose,
		"stack_n": n,
		"initial_num_feature_maps": init_fm_dim,
		"training_ds_size": train_size,
		"steps_per_epoch": steps_per_epoch,
		"val_steps_per_epoch": val_steps_per_epoch,
		"num_epochs": epochs,
		"loss": loss,
		"optim": optimizer,
		"optim_learning_rate_schedule": lr_schedule,
		"optim_momentum": optimizer_momentum,
		"optim_additional_metrics": optimizer_additional_metrics,
		"initializer": initializer,
		"callbacks": callbacks,
		"shortcut_type": shortcut_type
	}

  return config

- `3 (input_train, _), (_,_) = load_dataset()` - `cifar10.load_data()` returns `(train_data, train_labels), (test_data, test_labels)`. `input_train` holds the 49k (32x32x3) colour images.

- `5 # Generic configuration` - It is best to manually tweaked the following sizes based on the dataset being used (cifar10 has 32x32x3 image.shape) instead of using dynamic methods to <u>ensure consistency across different samples and allowing control to optimize computation</u>.
 - `9 validation_split = 0.1` - *10%* of input_train is used for validation.
 - `10 verbose = 1` - keras run in verbose mode; output are shown in terminal.
- `11 n = 3` -  number of residual block groups
  - relates to number of layers present in ResNet.
  - `n = 9` - equates *6n + 2 = 56* layers, ResNet56 implementation.
- `12 init_fm_dim = 16` - initial number of feature maps. He et al. choose an initial value of 16 feature maps, which increases by a factor of 2 when feature map size is halved.
- `13 shortcut_type = "identity"` - There are 2 shortcut types described in He et al:
  - `identity` and `projection`
  - They used identity shortcut for CIFAR-10 experiments, and identity shortcut may perform better than projection. This option can be changed easily.
- `16 - 23` - Using the <u>size</u> of training and validation datasets, <u>number of steps per epoch</u> is computed.
  - This is another design decision made in the He et al. paper. They trained their ResNet with 64000 iters. With this maximum, we compute the no. of steps/epoch for the data, as well as no. of epochs.

---------------------------------
Defining hyperparameters of loss function, learning rate, initializer, optimizer:
- `26 loss = CategoricalCrossEntropy(from_logits=True)`
  - Stabdard loss function for multiclass classification problems.
  - `from_logits=True` - Values of loss obtained are not normalized.
- `29-31 lr_scheduler`
  - Schedules to change learning rate after a few iterations.
  - Initially begins with 0.1 - high learning rate to get to conversion quickly
  - Divide by 10 (0.01) after 32000 iters (50%) and another by 75% training completion.
  - This is done through TensorFlow's `PieceWiseConstantDecay`.
- `34 initializer = tensorflow.keras.initializers.HeNormal()`
  - HeNormal initialization is used as per He et al.
- `37-39 optimizer`
  - He et al. used Stochastic Gradient Descent (SGD) with learning rate schedule above. They also used momentum term of 0.9 and weight decay of 0.0001.
  - (Author of code) mentions attempt to usd SGD with weight decay did not work, similar to ADAM. But default SGD with momentum without decay worked similarly.
- Callbacks to visualize and Checkpoint to save.
- Configuration is returned as Python dictionary.

###Load dataset

cifar10.load_data()
- Returns (input_train, label_train), (input_test, label_test)

In [ ]:
def load_dataset():
  return cifar10.load_data()

### Preprocess data

As described in the He et al. paper

1. per-pixel mean subtraction

2. Data augmentation
- Add 4 pixels of padding on all sides.
- Random sampling a 32 x 32 pixel crop from the padded image or its horizontal flip.

Note:
- TensorFlow's data aug options do not have random crop
- Main author in git, uses Jung (2018)'s proposed method (https://jkjung-avt.github.io/keras-image-cropping/) to generate random crops of a specific size from an input image.

In [ ]:
def random_crop(img, target_size):
  """
  @param
    target_size: list with (length, length)
  """
  # Note: image_data_format is 'channel_last'
  # SOURCE: https://jkjung-avt.github.io/keras-image-cropping/
  # Ensure image being processed has 3 colour channels (RGB image)
  # Will give AssertionError if not.
  assert img.shape[2] == 3
  height, width = img.shape[0], img.shape[1]
  th, tw = target_size
  x1 = np.random.randint(0, width-tw+1)
  y1 = np.random.randint(0, height-th+1)
  return img[y1:(y1+th), x1:(x1+tw),:]

def crop_generator(batches, crop_length):
  """
  Generate random crops from the image batches generated by the original iterator.
  SOURCE: https://jkjung-avt.github.io/keras-image-cropping/

  @param
    batches: Input a Keras ImageGen (Iterator)
  """
  while True:
    batch_x, batch_y = next(batches)
    batch_crops = np.zeros((batch_x.shape[0], crop_length, crop_length, 3))
    for i in range(batch_x.shape[0]):
      batch_crops[i] = random_crop(batch_x[i], (crop_length, crop_length))
    yield (batch_crops, batch_y)

Following will be the preprocessed_dataset()

In [ ]:
def preprocessed_dataset():
  """
  Load and preprocess and CIFAR-10 dataset.
  """
  (input_train, label_train), (input_test, label_test) = load_dataset()

  # Retrieve shape from model configuration and unpack into components
  config = model_configuration()
  height, width, dim = config.get("height"), config.get("width"), config.get("dim")
  num_classes = config.get("num_classes")

  # Data augmentation: perform zero padding on datasets
  paddings = tf.constant([[0,0,], [4,4], [4,4], [0,0]])
  input_train = tf.pad(input_train, paddings, mode="CONSTANT")

  # Convert scalar targets to categorical ones
  label_train = tf.keras.utils.to_categorical(label_train, num_classes)
  label_test = tf.keras.utils.to_categorical(label_test, num_classes)

  # Data generator for training data
  train_generator = tf.keras.preprocessing.image.ImageDataGenerator(
      validation_split = config.get("validation_split"),
      horizontal_flip = True,
      rescale = 1./255,
      preprocessing_function = tf.keras.applications.resnet50.preprocess_input
  )

  # Generate training and validation batches
  train_batches = train_generator.flow(input_train, label_train, batch_size=config.get("batch_size"), subset="training")
  valid_batches = train_generator.flow(input_train, label_train, batch_size=config.get("batch_size"), subset="validation")

  train_batches = crop_generator(train_batches, config.get("height"))
  valid_batches = crop_generator(valid_batches, config.get("height"))

  test_generator = tf.keras.preprocessing.image.ImageDataGenerator(
      preprocessing_function = tf.keras.applications.resnet50.preprocess_input,
      rescale=1./255)

  # Generate test batches
  test_batches = test_generator.flow(input_test, label_test, batch_size=config.get("batch_size"))

  return train_batches, valid_batches,  test_batches

Explanation:
- `5` - First load_dataset() loads CIFAR-10 dataset.
- `8-10` - Configuration options are retrieved from our previous implemented configuration dictionary.
- `13-14` - Pad 4 pixels on each side in the 2nd and 3rd dimension of input data.
  - TF uses channels-last strategy: (batch_size, rows, cols, channels).
  - Only rows and columns have added 4 pads.
- `17-18 label_train = tensorflow.keras.utils.to_categorical(label_train, num_classes)`
  - Convert scalar targets (integer values) into categorical format by means of <u>one_hot encoding</u>. This way, you'll be able to use categorical crossentropy loss.
  - Basically changes training labels into the 10 classes that CIFAR-10 has.
  - CIFAR10 dataset' label are integers ranging from 0 to 9. (E.g. Class 0: cat, Class 1: dog)
  - Suppose `label_train = [0, 2, 4]`, to_categorical turns it to:
  ```
  [[1, 0, 0, 0, 0, 0, 0, 0, 0, 0] # Class 0: Cat
   [0, 0, 1, 0, 0, 0, 0, 0, 0, 0] # Class 2: Bird
   [0, 0, 0, 0, 1, 0, 0, 0, 0, 0] # Class 4: Cow
  ```
- `21-26` - Define data generator for training data.
  - Data generator using generator principle (https://machinecurve.com/index.php/2020/04/06/using-simple-generators-to-flow-data-from-file-with-keras).
    - On-the-Fly Data Generation: Generate data in small batches or chunks when needed instead of loading all the data into memory at once.
    - Real-Time Augmentation: Apply transformation immediately during trianing, which may help improve model's generalization capabilities.
    - Efficient Memory Usage.
  - `validation_split` - splits part of the training set for validation.
  - `horizontal_flip` - Data aug design in He et al. paper to the padded input image before random cropping.
  - `rescale=1./255` - Performed to ensure <u>gradients do not explode</u>.
  - Finally, preprocess with TF's default Resnet.
    - This makes the input data follow a specific format (ResNet). E.g.:
      - Pixel values normalized to range [-1, 1].
      - Pixel values centred around zero (mean=0) with respect to ImageNet dataset, without scaling.
      - Convert from RGB to BGR
    - This preprocess handles the normalization and transformation.
  - From this `train_generator`, training and validation batches are generated.
    - `.flow` will flow the training data to data generator (taking only training or validation part depending on subset configuration).
    - Then `crop_generator` is used to convert the batches (which are 40x40 padded and possibly flipped images) to 32x32 format again (via random crop)
  - Similarly to `test_generation` without the flipping and padding/cropping (according to He et al. paper)
  - Training, validation and test batches are returned.
> For testing, we only evaluate the single view of the original 32 x 32 image. (He et al., 2016)

### Create the Residual Block

Residual block is composed of 2 methods:
- Regular mapping.
- A skip connection.

Using the Functional API, these paths can be created and merged.

In [ ]:
def residual_block(x, number_of_filters, match_filter_size=False):
  # Retrieve initializer
  config = model_configuration()
  initializer = config.get("initializer")

  # Create Skip connection
  x_skip = x

  # Perform the original mapping
  if match_filter_size:
    x = Conv2D(number_of_filters, kernel_size=(3,3), strides=(2,2),\
               kernel_initializer=initializer, padding="same")(x_skip)
  else:
    x = Conv2D(number_of_filters, kernel_size=(3,3), strides=(1,1),\
               kernel_initializer=initializer, padding="same")(x_skip)
  x = BatchNormalization(axis=3)(x)
  x = Activation("relu")(x)
  x = Conv2D(number_of_filters, kernel_size=(3,3),\
             kernel_initializer=initializer, padding="same")(x)

  # Perform matching of filter numbers if necessary
  if match_filter_size and config.get("shortcut_type")=="identity":
    x_skip = Lambda(lambda x: tf.pad(x[:, ::2, ::2, :], tf.constant([[0, 0,], [0, 0], [0, 0], [number_of_filters//4, number_of_filters//4]]), mode="CONSTANT"))(x_skip)
  elif match_filter_size and config.get("shortcut_type") == "projection":
    x_skip = Conv2D(number_of_filters, kernel_size=(1,1),\
			kernel_initializer=initializer, strides=(2,2))(x_skip)

  # Add the skip connection to the regular mapping
  x = Add()([x, x_skip])

	# Nonlinearly activate the result
  x = Activation("relu")(x)

	# Return the result
  return x

In the definition,
1. `3-4` - Load the initializer from the model configuration.
  - This is needed for initializing the `Conv2D` layers.
2. `7` - Create skip connection `x_skip` based on the input `x`.
  - This variable will be re-added to the output of residual block, creating the skip connection.
3. `9-15` - Original mapping is done.
  - (He et al.) mention each residual block is composed of 2 convolutional layers with 3x3 kernel size.
  - Depending on whether the size of first `Conv2D` layer is match by size to output filter map (which has lower amount), different stride size will be used.
> Then we use a stack of 6n layers with 3x3 convolutions on the feature maps of size (32, 16, 8) respectively, with 2n layers for each feature map size. (He et al., 2016)

4. `16-17` - Each layer is followed by Batch Normalization and a ReLU activation functions.
5. `18-29` - Add the skip connection.
  - This is done by `Add()`.
  - However, sometimes, the number of filters in `x` no longer match the number of filters in `x_skip` - _happens because no. of feature maps is increased with each group of residual blocks_.
    - To overcome this issue:
    > (A) zero-padding shortcuts are used for increasing dimensions, and all shortcuts are parameter free (the same as Table 2 and Fig. 4 right);
    >
    > (B) projection shortcuts are used for increasing dimensions, and other shortcuts are identity;
    >
    > (C) all shortcuts are projections.
    > (He et al., 2016)
  
    - `identity shortcut` from (B) is implemented by <u>padding zeros to the left and right side of the channel dimension</u>, using `Lambda` layer.
      - This layer type essentially allows us to manipulate Tensors in any way, returning the result. It works as follows:
        - Of the input Tensor `x`, where the 2nd and 3rd dimensions (rows and cols) are reduced in size by a factor of 2,
        - apply `number_of_filters//4` to each side of the feature map size dimension; expand the number of filters by 2.
        - This is necessary for the next group of residual blocks but using 50% on each side.
    - `projection mapping` from (C) is implemented by using
      - a `Conv2D` with a `1x1` kernel size and `2 stride` for generating the projection.

  - As He et al. found identity mappings to work best, configuration is set to `identity` by default.
6. `32` - Finally, the combined output/skip connection is nonlinearly activated with ReLU before passed to the next residual block.

### Create ResidualBlocks structure

In [ ]:
def ResidualBlock(x):
  # Retrieve values
  config = model_configuration()

  # Set initial filter size
  filter_size = config.get("initial_num_feature_maps")

  # Paper: "Then we use a stack of 6n layers (...)
  #         with 2n layers for each feature map size."
  # 6n/2n = 3, there are always 3 groups
  for layer_group in range(3):

    # Each block has 2 weighted layers,
    # and each group has 2n such blocks,
    # 2n/2 = n blocks per group
    for block in range(config.get("stack_n")):

      # Perform filter size increase at every
      # first layer in the 2nd block onwards.
      # Apply ConV block for projecting the
      # skip connection.
      if layer_group > 0 and block == 0:
        filter_size *= 2
        x = residual_block(x, filter_size, match_filter_size=True)
      else:
        x = residual_block(x, filter_size)

  # Return final layer
  return x

ResidualBlock is the logic for specifying all of the residual blocks.
1. `3-6` - Retrieve model configuration, and from it the initial filter size.
2. `11` - Paper suggests that a ResNet is built using
  - a stack of `6n` layers with `2n` layers for each feature map size.
  - `6n` layers divided by `2n` layers for each feature map size, means there will be 3 `6n/2n=3` groups pf residual blocks, with 3 filter map sizes.
    - He et al. use filter map sizes of 16, 32 and 64, respectively.
  - Using a `for` loop to iterate over this number of groups.
3. `16` - Each block in the code has 2 weighted layers,
  - There are 2 Conv layers in `residual_block()`,
    - excluding the one from skip connection should a projection mapping is used.
  - And each group has `2n` layers (per the paper and defined in `residual_block()`.
  - This means that there will be `2n/2=n` blocks per group.
  - This is the reason there is another for loop, creating `n` blcoks.
4. `22-26`
  - If this is second layer group or higher,
  - and it's the first block within the group,
    - increase the filter size by a factor two (per the paper)
    - then specify the `residual_block`, instructing it to match filter size (by manipulating the input and the skip connection using the identity or projection mapping).
  
- E.g. with `n=3`, yields `6n = 6*3 = 18` layers in residual blocks
- and `2n = 2*3 = 6` layers per group.
- With 3 groups, this matches.
- With `n=3`, `6n+2 = 6*3+2 = 20` layers in the network. This is ResNet-20.

### Model base: stacking your building blocks

To finalize the model by specifying its base structure. Recall that a ResNet is composed of `6n+2` weighted layers, and that there are `6n` such layers so far.

> The first layer is 3x3 convolutions (...) The network ends with a global average pooling, a 10-way fully-connected layer, and softmax. (He et al., 2016)

In [ ]:
def model_base(shp):
  """
  Base structure of the model, with residual blocks attached.
  """
  # Get number of classes from model configuration
  config = model_configuration()
  initializer = model_configuration().get("initializer")

  # Define model structure
  # logits are returned because Softmax is pushed to loss function.
  inputs = Input(shape=shp)
  x = Conv2D(config.get("initial_num_feature_maps"), kernel_size=(3,3),\
		strides=(1,1), kernel_initializer=initializer, padding="same")(inputs)
  x = BatchNormalization()(x)
  x = Activation("relu")(x)
  x = ResidualBlock(x)
  x = GlobalAveragePooling2D()(x)
  x = Flatten()(x)
  outputs = Dense(config.get("num_classes"), kernel_initializer=initializer)(x)

  return inputs, outputs

1. `11` - Inputs to the neural network are passed to the `Input` layer.
  - This is a default Keras layer that is capable of picking up inputs served to the model.
2. `12-13` - Create the initial 3x3 kernel size `Conv2D` layer with the initial number of filter maps, a 1x1 stride, zeros padding if necessary and kernel initializer specified in the model configuration (as per He et al.)
3. `14-15` - BatchNormalization and ReLU activation.
4. `16` - Let the input pass through the `6n` `ResidualBlocks` that is created above.
5. `17` - Data flows through `GlobalAveragePooling2D` nonweighted layer, performing global average pooling.
6. `18-19` - Data is flattened to be processed by a fully-connected layer (`Dense` layer), also initialized using He initialization.
  - This outputs a `(num_classes, )` shaped logits Tensor,
  - whihch is the case of CIFAR-10 is `(10, )` because of `num_classes = 10`.
7. `21` - References to `inputs` and `outputs` are returned so that the model can be initialized.

### Model initialization

This requires layer structure.
1. `9-10` - Simply call `model_base` definition using some input parameters representing input sample shape `shp`, and assign its outputs to `inputs, outputs`.
2. `13` - Now that layer references is gotten, the model can be initialized of the Keras `Model` class. The inputs and outputs can be specified and give it name, such as `resnet` (per the model configuration).
3. `14` - Compile the model with `model.compile` using loss function, optimizer and additional metrics configured in the model configuration, print the model summary, and return the model.

- Model is ready to be trained.

In [ ]:
def init_model():
  """
  Initialze a compiled ResNet model.
  """
  # Get shape from model configuration
  config = model_configuration()

  # Get model base
  inputs, outputs = model_base((config.get("width"), config.get("height"),\
                                config.get("dim")))

  # Initialize and compile model
  model = Model(inputs, outputs, name=config.get("name"))
  model.compile(loss=config.get("loss"),\
                optimizer=config.get("optim"),\
                metrics=config.get("optim_additional_metrics"))

  # Print model summary
  model.summary()

  return model

### Model Training

Keras has a high-level API to train models.
Training and validation batches are made with `ImageDataGenerator`. Simply pass the batches into the model.

This will begin the training process and return the trained `model` for evaluation.

In [ ]:
def train_model(model, train_batches, validation_batches):
    config = model_configuration()

    model.fit(train_batches,
              batch_size=config.get("batch_size"),
	          	epochs=config.get("num_epochs"),
	          	verbose=config.get("verbose"),
	          	callbacks=config.get("callbacks"),
	          	steps_per_epoch=config.get("steps_per_epoch"),
	          	validation_data=validation_batches,
	          	validation_steps=config.get("val_steps_per_epoch"))

    return model

### Model Evaluation
Simply use `model.evaluate` to output the test scores.

In [ ]:
def evaluate_model(model, test_batches):
	# Evaluate model
	score = model.evaluate(test_batches, verbose=0)
	print(f'Test loss: {score[0]} / Test accuracy: {score[1]}')

###Summary,
The following building blocks are made:
- Load data
- Build model layer structure
- Initialize model
- Training and evaluation of model

In the following `training_process`
1. Retrieve data of training, validation and testing batches using `preprocessed_dataset()`
2. Initialize a compiled ResNet model with `init_model()`
3. Train the model using training and validation batches with `train_model()`
4. Evaluate the model with `evaluate_model()`

In [ ]:
def training_process():
  # Get dataset
  train_batches, validation_batches, test_batches = preprocessed_dataset()

  # Initialize ResNet
  resnet = init_model()

  # Train ResNet model
  trained_resnet = train_model(resnet, train_batches, validation_batches)

  # Evaluate trained ResNet model post training
  evaluate_model(trained_resnet, test_batches)

###Begin script

In [ ]:
if __name__ == "__main__":
  training_process()

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)              ┃ Output Shape           ┃        Param # ┃ Connected to           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)  │ (None, 32, 32, 3)      │              0 │ -                      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d (Conv2D)           │ (None, 32, 32, 16)     │            448 │ input_layer[0][0]      │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization       │ (None, 32, 32, 16)     │             64 │ conv2d[0][0]           │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation (Activation)   │ (None, 32, 32, 16)     │              0 │ batch_normalization[0… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_1 (Conv2D)         │ (None, 32, 32, 16)     │          2,320 │ activation[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_1     │ (None, 32, 32, 16)     │             64 │ conv2d_1[0][0]         │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_1 (Activation) │ (None, 32, 32, 16)     │              0 │ batch_normalization_1… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_2 (Conv2D)         │ (None, 32, 32, 16)     │          2,320 │ activation_1[0][0]     │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add (Add)                 │ (None, 32, 32, 16)     │              0 │ conv2d_2[0][0],        │
│                           │                        │                │ activation[0][0]       │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_2 (Activation) │ (None, 32, 32, 16)     │              0 │ add[0][0]              │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_3 (Conv2D)         │ (None, 32, 32, 16)     │          2,320 │ activation_2[0][0]     │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ batch_normalization_2     │ (None, 32, 32, 16)     │             64 │ conv2d_3[0][0]         │
│ (BatchNormalization)      │                        │                │                        │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_3 (Activation) │ (None, 32, 32, 16)     │              0 │ batch_normalization_2… │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_4 (Conv2D)         │ (None, 32, 32, 16)     │          2,320 │ activation_3[0][0]     │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ add_1 (Add)               │ (None, 32, 32, 16)     │              0 │ conv2d_4[0][0],        │
│                           │                        │                │ activation_2[0][0]     │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ activation_4 (Activation) │ (None, 32, 32, 16)     │              0 │ add_1[0][0]            │
├───────────────────────────┼────────────────────────┼────────────────┼────────────────────────┤
│ conv2d_5 (Conv2D)         │ (None, 32, 32, 16)     │          2,320 │ activation_4[0][0]     │
├──────────────────────

 Total params: 855,082 (3.26 MB)

 Trainable params: 853,034 (3.25 MB)

...

Epoch 1/182
351/351 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.1586 - loss: 3.0044

351/351 ━━━━━━━━━━━━━━━━━━━━ 1611s 5s/step - accuracy: 0.1587 - loss: 3.0023 - val_accuracy: 0.2188 - val_loss: 1.9954
Epoch 2/182
351/351 ━━━━━━━━━━━━━━━━━━━━ 1608s 5s/step - accuracy: 0.2634 - loss: 1.8946 - val_accuracy: 0.1164 - val_loss: 2.9755
Epoch 3/182
351/351 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.3351 - loss: 1.7369

351/351 ━━━━━━━━━━━━━━━━━━━━ 1595s 5s/step - accuracy: 0.3351 - loss: 1.7368 - val_accuracy: 0.3354 - val_loss: 1.8091
Epoch 4/182
351/351 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.3778 - loss: 1.6462

351/351 ━━━━━━━━━━━━━━━━━━━━ 1578s 4s/step - accuracy: 0.3778 - loss: 1.6461 - val_accuracy: 0.3352 - val_loss: 1.7619
Epoch 5/182
195/351 ━━━━━━━━━━━━━━━━━━━━ 11:39 4s/step - accuracy: 0.4150 - loss: 1.5635

KeyboardInterrupt: 